# Welcome to Week 4 - LangChain and LangGraph

## Lab 1: Abstraction Levels and the building blocks

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The LangChain docs</h2>
            <span style="color:#00bfff;">The documentation lives at <a href="https://docs.langchain.com/oss/python/">https://docs.langchain.com/oss/python/</a>. It is worth a browse. Watch out for older 0.x material that still floats around the web, because the modern API is quite different.
            </span>
        </td>
    </tr>
</table>

## The four levels of abstraction

LangChain and LangGraph form 4 levels of abstraction, with each one built on the ones before. The terminology is confusing because 'LangChain' appears in a few places..

| Layer | Packages | What it gives you | What you control |
|---|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | chat models, the `@tool` decorator, messages, structured output | everything, including the tool loop by hand |
| 2. Orchestration | `langgraph` | a graph of steps, with state, memory and checkpointing | the control flow (you design the graph) |
| 3. Agent | `langchain` (`create_agent`) | the standard agent loop, prebuilt | just model, tools and a prompt |
| 4. Harness | `deepagents` (`create_deep_agent`) | an opinionated harness with planning, sub-agents and a filesystem | your intent |


## Order of play for this week:

DAY 1 (today): The building blocks  
DAY 2: LangGraph  
DAY 3: LangChain create_agent  
DAY 4: Deep Agents  
DAY 5: The Sidekick project  

## Today is Layer 1: the building blocks

This is where LangChain began: an abstraction layer, not unlike LiteLLM - but a more heavyweight version.

In [22]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_community.tools import GoogleSerperRun



load_dotenv(override=True)

C:\Users\krsna\AppData\Local\Temp\ipykernel_23208\881870918.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import GoogleSerperAPIWrapper


True

In [4]:
!uv add ChatOllama

Resolved 245 packages in 5.87s
Prepared 2 packages in 392ms
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 2 packages in 57ms
 + asyncio==4.0.0
 + chatollama==0.2.13


### A first model call

`ChatOpenAI` is the abstraction around OpenAI calls, and there are similar packages for other LLM providers.

We call `invoke` with a prompt; this is a key method in LangChain.

In [6]:
# llm = ChatOpenAI(model="gpt-5.4-mini")

llm_llama = ChatOllama(
    model="llama3.2:latest",
    temperature=0.7,
    base_url="http://localhost:11434" # default
)


message = "In 1 sentence, what does it mean for an AI Agent to be autonomous"

reply = llm_llama.invoke(message)

print(reply.content)

An AI agent is considered autonomous when it has the ability to make decisions and take actions without being explicitly programmed or controlled by a human, operating based on its own internal logic, rules, and objectives.


### Streaming

For a live, token by token feel, swap `invoke` for `stream` and loop over the chunks

In [8]:
for chunk in llm_llama.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)

Here is a two-line poem about autonomous agents:

"Free from human hand, they roam and thrive,
Their logic guiding them, as they navigate and strive."

### Any OpenAI-compatible provider

As before; we can use OpenAI compatible endpoints with the ChatOpenAI object

In [9]:
openrouter_llm = ChatOpenAI(
    model="anthropic/claude-haiku-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

reply = llm_llama.invoke("In one sentence, what is LangChain?")
print(reply.content)

LangChain is an open-source, blockchain-based framework for building scalable, modular, and interoperable applications, allowing developers to create custom blockchain-based solutions by connecting different blockchain networks and enabling seamless data sharing and interactions.


### Messages

LangChain comes with abstractions around SystemMessage, HumanMessage, AIMessage, although you can use the usual list-of-dicts instead.

In [11]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]

print(llm_llama.invoke(messages).content)

# The exact same call using plain dictionaries, the format you already know
messages_as_dicts = [
    {"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
    {"role": "user", "content": "What is the capital of France?"},
]
print(llm_llama.invoke(messages_as_dicts).content)

Paris is the capital France.
Paris is the official capital.


### Tools with the `@tool` decorator

A tool is a Python function the model is allowed to call. The modern way to make one is the `@tool` decorator. Your docstring becomes the description the model reads, and your type hints become the argument schema, just like `@function_tool` with OpenAI Agents SDK.

This replaces the older `Tool(...)` wrapper from earlier versions of LangChain.

In [12]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("args:", get_share_price.args)
print("called directly:", get_share_price.invoke({"symbol": "AAPL"}))

name: get_share_price
description: Return the current share price for a given ticker symbol.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 241.5


### Giving tools to the model

This is a bit clunky; redemption will come in Day 3 when we use the create_agent features,

For now, we have to write the loop ourselves, rather like we did in Week 1.

The first step is to bind the tools to the model with `bind_tools`. Now when we invoke, the model may come back not with an answer but with a request to run a tool. That request shows up in `.tool_calls`.

In [13]:
llm_with_tools = llm_llama.bind_tools([get_share_price])

response = llm_with_tools.invoke("What is the share price of {'symbol': 'AAPL'}")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AAPL'}, 'id': '2cfddd97-af08-4e61-9294-0b31f4ec542b', 'type': 'tool_call'}]


### Running the tool loop by hand

So now we need to write a little loop, quite similar to Week 1

In [16]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the share price of Apple?")]
message = llm_llama.invoke(conversation)
print(f"The share price of {conversation[0].content} without tools:", message)
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        print(f"The share price of {call['args']} with tools:", result)
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

The share price of What is the share price of Apple? without tools: content="I'm not able to provide real-time data, but I can suggest some ways for you to find the current share price of Apple.\n\nYou can check the current share price of Apple on various financial websites, such as:\n\n* Yahoo Finance: [finance.yahoo.com/quote/AAPL/](http://finance.yahoo.com/quote/AAPL/)\n* Google Finance: [finance.google.com/quote/AAPL/](http://finance.google.com/quote/AAPL/)\n* Bloomberg: [bloomberg.com/markets/stocks/quote/AAPL/](http://bloomberg.com/markets/stocks/quote/AAPL/)\n* Apple's official website: [investors.apple.com/](http://investors.apple.com/)\n\nPlease note that share prices can fluctuate rapidly and may not be up-to-date. It's always a good idea to check multiple sources for the most current information." additional_kwargs={} response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-10-11T08:00:44.2379543Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11610327600

### Structured output

Similar to OpenAI Agents SDK, we can require the model to respond with a Pydantic subclass

In [17]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm_llama.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Reliance the Indian Conglomerate company.")
print(company)
print("Just the ticker:", company.ticker)

name='Reliance Industries Limited' ticker='RIL' founded_year=1973
Just the ticker: RIL


## That is Layer 1

It's like a more rich and more involved version of LiteLLM.

In [18]:
@tool
def get_weather_report(symbol: str) -> float:
    """Return the current weather report for a given location."""
    fake_weather = {"New York": 72.5, "Los Angeles": 80.0, "Chicago": 65.0}
    return fake_weather.get(symbol, 0.0)

print("name:", get_weather_report.name)
print("description:", get_weather_report.description)
print("args:", get_weather_report.args)
print("called directly:", get_weather_report.invoke({"symbol": "New York"}))

name: get_weather_report
description: Return the current weather report for a given location.
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly: 72.5


In [ ]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

llm_with_tools = llm_llama.bind_tools([get_weather_report, search])

response = llm_with_tools.invoke("What is the weather like in New York? Compare it with real time weather of new york using the search tool.")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: ''
tool_calls: [{'name': 'get_weather_report', 'args': {'symbol': 'New York'}, 'id': '2b881ced-3a90-47c5-8767-cf1de941f043', 'type': 'tool_call'}, {'name': 'google_serper', 'args': {'query': 'New York real time weather'}, 'id': 'fa35b85f-ab07-4266-9d14-0bcda6fe85b1', 'type': 'tool_call'}]


In [29]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the weather like in New York? Compare it with real time weather of new york using the search tool.")]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_weather_report":
        result = get_weather_report.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        print(f"The get_weather_report tool called:", str(result))
    elif call["name"] == "google_serper":
        result = search.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        print(f"The search tool called:", str(result))

# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

The get_weather_report tool called: 72.5
The search tool called: AccuWeather · Feels Like. 59°. Sunrise. 6:58 AM · Humidity. 78%. Sunset. 6:27 PM · Windspeed. SSW 6 mph. Moonrise. 3:35 AM · Pressure. 29.84 in. Moonset. 4:58 PM. Current Hazards · Briefing Page · River Flooding · Current Conditions · Observations (Map) · Satellite · Radar · Local KOKX/KJFK/KEWR Enhanced Radars · Forecasts. The current RealFeel® Temperature is 61° (Cool) based on observed conditions including 66% humidity, 77% cloud cover, a UV index of 0, and winds of 3 mph. In ... New York City Weather Forecasts. Weather Underground provides local & long-range weather forecasts, weatherreports, maps & tropical weather conditions for ... Forecast conditionsSunshine and clouds mixed. High 60F. Winds NW at 10 to 15 mph. Weather stories. Today's and tonight's New York City, New York weather forecast, weather conditions and Doppler radar from The Weather Channel and Weather.com. Enjoy comfortable conditions across the New Yo

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Write a second tool of your own, perhaps one that looks up a fictional weather report for a city. Bind both tools to the model and ask a question that needs both, then run the tool loop by hand until the model gives a final answer. It will feel more gritty than OpenAI Agents SDK or CrewAI.
            We will fix that by Day 3!
            </span>
        </td>
    </tr>
</table>